In [ ]:
pip install transformers scikit-learn torch numpy pandas peft ollama

In [1]:
#CALL MODEL

#choose parent model
model_name = "llama3.1:8b-instruct-q8_0"

#pull mode
!ollama pull {model_name}

!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling cc04e85e1f86: 100% ▕██████████████████▏ 8.5 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 4a4a958ae550: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
NAME                                                  ID              SIZE      MODIFIED               
llama3.1:8b-instruct-q8_0                             b158ded76fa0    8.5 GB    Less than a second ago    
gpt-oss:20b                                           f2b8351c629c    13 GB     4 days ago                
llama3.1:8b-instruct-q

In [ ]:
#LOAD DATA AND PACKAGES

#import packages and datasets
import pandas as pd
import numpy as np
import sklearn as sk
import json
import ollama
from pathlib import Path
from typing import Sequence
import re

#Unlabelled dataset
df_test = pd.read_csv("unlabelled_frames_test.csv")
#Labelled dataset
df_train = pd.read_csv('labelled_frames_train.csv')
df_test_lab = pd.read_csv('labelled_frames_test.csv')
df_gold = pd.concat([df_train, df_test_lab], axis=0).reset_index(drop=True)
#Frame names
frame_names = df_gold.columns[7:14].to_list()
#Codebook
with open('framing_codebook_short.json', 'r') as f:
    codebook = json.load(f)
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)


In [3]:
#HELPER FUNCTIONS

##parsing responses from model
def parse_json_with_fallback(content_str, frame_names):
    #Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'

    # Now try parsing
    try:
        #if the response can be parsed
        dict_val = json.loads(content_str)
        #remove any leading or trailing white spaces in the keys
        cleaned_dict = {k.strip(): v for k, v in dict_val.items()}

        annotations = []

        #parse response for each frame
        for f in frame_names:
            try:
                f_val = str(cleaned_dict.get(f))

                binary_val = pd.NA
                if re.search("yes",f_val.strip().lower()):
                    binary_val = 1
                if re.search("no",f_val.strip().lower()):
                    binary_val = 0

            except:
                binary_val = pd.NA
            
            annotations.append(binary_val)

        return annotations
    
    except json.JSONDecodeError:
        return [pd.NA]*len(frame_names)



##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner', suffixes = ('_pred', '_gold'))   # keep only matching IDs
            .dropna(subset=[column_pred + '_pred', column_gold + '_pred'])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred + '_pred'] = df_temp[column_pred + '_pred'].astype(int)
    df_temp[column_gold + '_gold'] = df_temp[column_gold + '_gold'].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    precision = round(sk.metrics.precision_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    recall = round(sk.metrics.recall_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)

    print("Accuracy for " + column_gold +  " is " + str(acc) + "%, and Cohen's kappa is " + str(k))

    return([acc, k, f1, precision, recall])

#formatting fine tuning dataset to JSONL and saving
def to_ollama_jsonl(df_in: pd.DataFrame,
                    out_path: str | Path,
                    system_prompt: str,
                    user_prompt: str,
                    frame_names: Sequence[str],) -> Path:
    """
    Convert <article text, label> rows into Ollama‑ready JSONL.
    Each line is one conversation:  system ➜ user ➜ assistant
    """
    out_path = Path(out_path)
    with out_path.open("w", encoding="utf‑8") as f:
        for _, row in df_in.iterrows():
            answer = {}
            for frame in frame_names:
                label = row[frame]

                answer[frame] = "yes" if label == 1 else "no"

            record = {
                "messages": [
                    {"role": "system",    "content": system_prompt},
                    {"role": "user",      "content": user_prompt + row["text"]},
                    {"role": "assistant", "content": answer},
                ]
            }
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
    return out_path

#prompt builder for one to many classifier
def prompt_maker(frame):
    """ 
    Makes system and user level prompts for each frame 
    """

    codebook_str_f = json.dumps(codebook[frame_names.index(frame)], indent=2, ensure_ascii=False)

    SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
    You're only interested in studying the presence or absence of the frame: """ + frame + """. Here is the codebook definition of the frame: """ + codebook_str_f

    USER_PROMPT = """Identify if the frame (""" + frame + """) is present in the news article using the following guidelines:
    1. Read the entire article carefully before coding
    3. Some articles maybe irrelevant to the Mpox epidemic. In this case, mark "no".  
    4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
    5. Use frame description and examples to guide decisions
    6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

    Provide your response in a JSON array format, as follows, and include nothing else in the response: {" """ + frame + """ ": "yes/no"}.

    If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". 
    
    Here's the article: """

    return [SYSTEM_PROMPT, USER_PROMPT]

In [ ]:
#ZERO SHOT: ONE-TO-MANY

#ONE-TO-MANY PROMPTS
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decision
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format, as follows, and include nothing else in the response: 
{"sexual stigma and transmission routes": "yes/no",
"racial disparities and stigmatising name": "yes/no",
"global relations": "yes/no",
"public health failure": "yes/no",
"epidemic preparedness and surveillance": "yes/no",
"human-interest stories": "yes/no",
"broader health issues": "yes/no"}.

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """


#sample for testing things first
#df_sample = df_test.sample(n= 30, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
df1[frame_names] = pd.NA

for j in range(len(predictions)):
    content_str = predictions[j].lower()
    annotation = parse_json_with_fallback(content_str, frame_names)
    df1.loc[j, frame_names] = annotation
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
#save files
#scores_df.to_csv("Predicted/" + model_name + "_scores.csv")
df1.to_csv("Predicted/" + model_name + "_zshot.csv")

In [ ]:
#ZERO SHOT: ONE-TO-ONE

#sample for testing things first
#df_sample = df_test.sample(n= 30, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()

df1 = df_sample

for f in frame_names:
    prompts = prompt_maker(f)
    SYSTEM_PROMPT = prompts[0]
    USER_PROMPT = prompts[1]

    predictions = list()

    for i in range(len(df1)):
        try:
            messages = [
            {"role": "system", 
            "content": SYSTEM_PROMPT
            },

            {
            "role": "user", 
            "content": USER_PROMPT + df1["text"][i] 
            }

                ]
        
            outputs = ollama.chat(model= model_name, messages= messages)

            predictions.append(outputs.message.content)
        
        #if there is an error
        except Exception as e:
                predictions.append(None)
        
        #stream progress
        if(i%10 == 0): print(str(i) + " iterations finished")


    #save the responses
    df1[f] = pd.NA

    for j in range(len(predictions)):
        content_str = predictions[j].lower()
        annotation = parse_json_with_fallback(content_str, [f])
        df1.loc[j, f] = annotation[0]
    
    print(f + ' annotations finished')
    
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
#save files
#scores_df.to_csv("Predicted/" + model_name + "_1to1_scores.csv")
df1.to_csv("Predicted/" + model_name + "_zshot_1to1.csv")

In [ ]:
#FINE TUNED: ONE-TO-MANY 

#ONE-TO-MANY PROMPTS

SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decisions
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format, as follows, and include nothing else in the response: 
{"sexual stigma and transmission routes": "yes/no",
"racial disparities and stigmatising name": "yes/no",
"global relations": "yes/no",
"public health failure": "yes/no",
"epidemic preparedness and surveillance": "yes/no",
"human-interest stories": "yes/no",
"broader health issues": "yes/no"}.

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """

#fine-tune on all 400 articles
df_tune = df_train.copy()

#format the fine tuning data
tune_file = to_ollama_jsonl(df_tune, "Fine_tune/framing_tune.jsonl", SYSTEM_PROMPT, USER_PROMPT, frame_names)

#Let the fine tuning begin
ft_model_name = model_name + "_framing_ft"

client = ollama.Client()                       # defaults to http://localhost:11434
digest = client.create_blob(tune_file)         # uploads the JSONL, returns sha256
progress = client.create(
    model       = ft_model_name,     # new tag
    from_       = model_name,                  # parent model
    files       = {"framing_tune.jsonl": digest},
    parameters  = {"num_epochs": 3},           # any ggml‑compatible training args
    stream      = True                         # chunked progress
)
for chunk in progress:                         # stream shows download / training bar
    print(chunk.status, chunk.completed, "/", chunk.total)

#annotate test set with fine tuned model
#sample some for testing the code first
#df_sample = df_test.sample(n= 30, random_state= 42).reset_index(drop=True)
#otherwise
df_sample = df_test


df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= ft_model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
df1[frame_names] = pd.NA

for j in range(len(predictions)):
    content_str = predictions[j].lower()
    annotation = parse_json_with_fallback(content_str, frame_names)
    df1.loc[j, frame_names] = annotation
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    

#save files
#scores_df.to_csv("Predicted/" + ft_model_name + "_scores.csv")
df1.to_csv("Predicted/" + model_name + "_ft.csv")

In [ ]:
#FINE TUNED: ONE-TO-ONE

#fine tune on all 400 articles
df_tune = df_train

#sample for testing things first
#df_sample = df_test.sample(n= 30, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()

df1 = df_sample

for f in frame_names:
    #obtain prompts for that frame
    prompts = prompt_maker(f)
    SYSTEM_PROMPT = prompts[0]
    USER_PROMPT = prompts[1]

    #fine tune models for each frame
    #format the fine tuning data for that frame
    tune_file = to_ollama_jsonl(df_tune, "Fine_tune/framing_tune_frame" + str(frame_names.index(f)) + ".jsonl", SYSTEM_PROMPT, USER_PROMPT, [f])

    #Let the fine tuning begin
    ft_model_name = model_name + "_framing_ft_frame" + str(frame_names.index(f))
    client = ollama.Client()                       # defaults to http://localhost:11434
    digest = client.create_blob(tune_file)         # uploads the JSONL, returns sha256
    progress = client.create(
        model       = ft_model_name,     # new tag
        from_       = model_name,                  # parent model
        files       = {"Fine_tune/framing_tune_frame" + str(frame_names.index(f)) + ".jsonl": digest},
        parameters  = {"num_epochs": 3},           # any ggml‑compatible training args
        stream      = True                         # chunked progress
    )
    for chunk in progress:                         # stream shows download / training bar
        print(chunk.status, chunk.completed, "/", chunk.total)
    

    predictions = list()

    for i in range(len(df1)):
        try:
            messages = [
            {"role": "system", 
            "content": SYSTEM_PROMPT
            },

            {
            "role": "user", 
            "content": USER_PROMPT + df1["text"][i] 
            }

                ]
        
            outputs = ollama.chat(model= ft_model_name, messages= messages)

            predictions.append(outputs.message.content)
        
        #if there is an error
        except Exception as e:
                predictions.append(None)
        
        #stream progress
        if(i%10 == 0): print(str(i) + " iterations finished")


    #save the responses
    df1[f] = pd.NA

    for j in range(len(predictions)):
        content_str = predictions[j].lower()
        annotation = parse_json_with_fallback(content_str, [f])
        df1.loc[j, f] = annotation[0]
    
    print(f + ' annotations finished')
    
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
#save files
#scores_df.to_csv("Predicted/" + model_name + "_ft_1to1_scores.csv")
df1.to_csv("Predicted/" + model_name + "_ft_1to1.csv")